# KNN — K 및 투표 가중치

113,920장 학습, 13개 고유 투표 조건.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("KNN", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
knn=model; tr=knn._fit_X; ty=knn._y; kbase=base
print("Local output:",root)

In [ ]:
def vote(d,ids,k,alpha,train_y=ty):
    ds=d[:,:k]; ls=train_y[ids[:,:k]]
    zero=ds==0
    weights=np.power(np.maximum(ds,1e-30),-alpha)
    weights=np.where(zero.any(axis=1)[:,None],zero,weights)
    scores=np.zeros((len(ds),16))
    np.add.at(scores,(np.arange(len(ds))[:,None],ls),weights)
    return np.argmax(scores,axis=1)


In [ ]:
dv,iv=knn.kneighbors(Xv,n_neighbors=9)
np.savez_compressed(root/"03_KNN"/"neighbors_handmade_p2_k9.npz",distance=dv,index=iv)

In [ ]:
print('baseline max9 vote mismatch',np.sum(vote(dv,iv,3,1)!=kbase))
krows=[]
kpreds={}
for k in [1,3,5,7,9]:
 for alpha in ([1] if k==1 else [.5,1,2]):
    name=f'k{k}_a{alpha}'
    pred=vote(dv,iv,k,alpha)
    kpreds[name]=pred
    krows.append({'condition':name,'k':k,'alpha':alpha,**stats(pred,kbase)})
kr=pd.DataFrame(krows)
kr.to_csv(root/'03_KNN/EXP-FT-KNN-002_handmade.csv',index=False)
pd.DataFrame({'true':y,**kpreds}).to_csv(root/'03_KNN/EXP-FT-KNN-002_predictions.csv',index=False)
print(kr[['condition','correct','macro_precision','macro_f1','corrected','regressed']].to_string(index=False))